Economics output, kWh saved for both winter and summer. hourly outputs for NEM integration

T_indoor_C is a fixed reference indoor temperature.

Transient model based off of finite volume method. 

# **Library Imports**

In [403]:
# dataclass lets us create simple classes without writing lots of setup code
from dataclasses import dataclass
# Path helps Python work with file paths more cleanly than plain strings
from pathlib import Path
# pandas is used to read and inspect CSV tables
import pandas as pd
# numpy gives us np.nan, which represents blank/missing values in tables + facilitates numerical operations on arrays
import numpy as np; import math
# for plotting
import matplotlib.pyplot as pl
# solving initial temps
import scipy.optimize as opt

# Importing input tables
- Table 1: Barra 2 Weather Dataset
- Table 2: Building Parameter Summary
- Table 3: Surface Geometry
- Table 4: Roof Layers

In [404]:
# pathway = "../Input Tables" -> Calling to folders on shared google drive 
INPUT_DIR = Path("../Input Tables")

# This function loads any input CSV table using the same process, so we don't have to repeat 
# the path checks, CSV import, and blank-cell cleanup for Tables 1-4
def load_input_table(file_name):
    csv_path = INPUT_DIR / file_name
    # Read the CSV into a pandas DataFrame
    df = pd.read_csv(csv_path)
    # Strip whitespace from column names (removes leading/trailing spaces)
    df.columns = df.columns.str.strip()
    # Turn blank cells into proper missing values so Python can handle them safely
    df = df.replace({"": np.nan})
    return df # Send the cleaned table back to the line that called this function

In [405]:
# ----------------------- TABLE 1: BARRA2 weather data ---------------------------
weather_table = load_input_table("barra2_-37.91_145.13_2007.csv")
# Convert UTC timestamp column into datetime format
weather_table["time_UTC"] = pd.to_datetime(weather_table["time_UTC"], utc=True)
# Convert UTC time to local Melbourne 24hr time and add to end of table
weather_table["time_Melbourne"] = weather_table["time_UTC"].dt.tz_convert("Australia/Melbourne").dt.tz_localize(None)
barra_table = weather_table.copy()#.iloc[145:]

# ----------------------- TABLE 2: Building summary ----------------------------
# buildings_table = load_input_table("buildings.csv")

# ----------------------- TABLE 3: Surface geometry ----------------------------
# surfaces_table = load_input_table("surfaces.csv")

# ----------------------- TABLE 4: Roof Layer properties ----------------------
roof_layers_table = load_input_table("Roof_layers.csv")
display(roof_layers_table)


,Material_type,Thickness_m,Density_Kg_m3,Spec_Heat_Cap_J_KgK,R_value_m2K_W,Absorptivity,Emissivity,Absorptivity_cool,Emissivity_cool
0,Slate,0.006000,2700.0,753,0.003000,0.90,0.9,0.140,0.880
1,Terracotta,0.014926,1900.0,800,0.017769,0.70,0.9,0.140,0.880
2,Concrete,0.017140,1900.0,837,0.008687,0.65,0.9,0.068,0.875
3,Steel_light,0.000420,7800.0,480,0.000008,0.30,0.9,0.068,0.875
4,Steel_dark,0.000420,7800.0,480,0.000008,0.80,0.9,0.068,0.875
5,Insulation_new,0.215000,12.0,840,4.100000,NaN,NaN,NaN,NaN
6,Insulation_old,0.130000,12.0,840,2.500000,NaN,NaN,NaN,NaN
7,Cavity,0.600000,1.2,1006,0.300000,NaN,NaN,NaN,NaN
8,Plaster,0.013000,950.0,840,0.081250,NaN,NaN,NaN,NaN


# Roofing Material Class

This section converts each row in Table 4 into material objects

In [406]:
@dataclass
class Roof_Layers:
    # Basic identifier from Table 5
    Material_type: str
    # Thermal properties from Table 5
    Thickness_m: float = None
    Density_Kg_m3: float = None
    Heat_capacity_J_KgK: float = None
    R_value_m2K_W: float = None
    Absorptivity: float = None
    Absorptivity_cool: float = None
    Emissivity: float = None
    Emissivity_cool: float = None
# This function converts one row from Roof_Layers into one roof object
def make_roof_from_row(row):
    return Roof_Layers(Material_type=row["Material_type"],Thickness_m=row["Thickness_m"],Density_Kg_m3=row["Density_Kg_m3"],Heat_capacity_J_KgK=row["Spec_Heat_Cap_J_KgK"],R_value_m2K_W=row["R_value_m2K_W"],Absorptivity = row["Absorptivity"],Absorptivity_cool = row["Absorptivity_cool"],Emissivity = row["Emissivity"], Emissivity_cool = row["Emissivity_cool"])
# This converts every row in Table 5 into a Roof_Layers object
roof_layers_table = {row["Material_type"]: make_roof_from_row(row)for _, row in roof_layers_table.iterrows()}
# testing bottom row to see if info has been pulled correctly 
roof_layers_table["Concrete"]


Roof_Layers(Material_type='Concrete', Thickness_m=0.017140115, Density_Kg_m3=1900.0, Heat_capacity_J_KgK=837, R_value_m2K_W=0.008686641, Absorptivity=0.65, Absorptivity_cool=0.068, Emissivity=0.9, Emissivity_cool=0.875)

# Defining House Parameters


In [407]:
# defining indoor boundary temperature condition 
T_indoor_C = 20
# Defining Roof height above ground - Later this should come from Table 2
z_roof_m = 9
# DEFINING PARAMETERS - LATER TO BE REPLACED BY INPUT TABLES
# Dummy roof parameters until data is available
roof_plan_area = 120 # the birdseye roof area (plan area) {m^2} 
roof_tilt = 20  # roof pitch (degrees)
roof_azimuth = 0  # direction roof faces
# Assumption: simple roof split into two equal roof planes
roof_planes = pd.DataFrame({
    "roof_plane_id": ["roof_plane_1","roof_plane_2"],
    # Bird's-eye/projected area for each half of the roof
    "plan_area": [roof_plan_area/2,roof_plan_area/2],
    # Roof pitch for each plane
    "tilt_deg": [roof_tilt,roof_tilt],
    # Direction each plane faces
    "azimuth_deg":[roof_azimuth,(roof_azimuth+180)%360],})
# Convert bird's-eye area into actual sloped roof surface area
roof_planes["actual_area"] = (roof_planes["plan_area"]/np.cos(np.radians(roof_planes["tilt_deg"])))
display(roof_planes)
# defining variable for the total actual roof area
actual_roof_area = roof_planes["actual_area"].sum()


,roof_plane_id,plan_area,tilt_deg,azimuth_deg,actual_area
0,roof_plane_1,60.0,20,0,63.850666
1,roof_plane_2,60.0,20,180,63.850666


# **Shortwave Radiation**

Note: Treating rsdsdir with solar incidence already included, thus dont need an angle factor 


In [408]:
# This function calculates absorbed shortwave radiation at the roof surface
# Outputs the absorbed shortwave radiation in W for the actual roof surface
def q_sw_absorbed(rsdsdir_Wm2, rsdsdif_Wm2, alpha):
    # Absorbed shortwave radiation for the plan roof area surface
    q_sw_W_m2 = alpha*(rsdsdir_Wm2+rsdsdif_Wm2) 
    return q_sw_W_m2
def q_sw_absorbed_pv(rsdsdif_Wm2, alpha):
    q_sw_pv = alpha*rsdsdif_Wm2
    return q_sw_pv 

# Copy weather table so we can add model outputs without changing the original weather data
roof_sw_table = barra_table.copy(); roof_sw_table = roof_sw_table.drop(columns=["time_UTC"])
# Store the roof areas in the output table so the calculation is easy to check later
roof_sw_table["actual_roof_area"] = actual_roof_area
# define roof types
roof_types = ["Slate","Terracotta","Concrete","Steel_light","Steel_dark"]
# calculate the absorbed shortwave radiation for each roof type
for roof_type in roof_types:
    roof = roof_layers_table[roof_type]
    alpha = roof.Absorptivity; alpha_cool = roof.Absorptivity_cool
    roof_sw_table[f"sw_absorbed_W_m2_{roof_type}"] = roof_sw_table.apply(lambda row: q_sw_absorbed(row["rsdsdir_Wm2"], row["rsdsdif_Wm2"], alpha), axis=1)
    roof_sw_table[f"sw_pv_W_m2_{roof_type}"] = roof_sw_table.apply(lambda row: q_sw_absorbed_pv(row["rsdsdif_Wm2"], alpha), axis=1)
    roof_sw_table[f"sw_absorbed_W_m2_{roof_type}_cool"] = roof_sw_table.apply(lambda row: q_sw_absorbed(row["rsdsdir_Wm2"], row["rsdsdif_Wm2"], alpha_cool), axis=1)
    roof_sw_table[f"sw_pv_W_m2_{roof_type}_cool"] = roof_sw_table.apply(lambda row: q_sw_absorbed_pv(row["rsdsdif_Wm2"], alpha_cool), axis=1)
# Show the first few hourly results
display(roof_sw_table.head())

,rsdsdir_Wm2,rsdsdif_Wm2,temp_K,rel_humidity_percent,wind_ms,rsds_total_Wm2,temp_C,time_Melbourne,actual_roof_area,sw_absorbed_W_m2_Slate,...,sw_absorbed_W_m2_Concrete_cool,sw_pv_W_m2_Concrete_cool,sw_absorbed_W_m2_Steel_light,sw_pv_W_m2_Steel_light,sw_absorbed_W_m2_Steel_light_cool,sw_pv_W_m2_Steel_light_cool,sw_absorbed_W_m2_Steel_dark,sw_pv_W_m2_Steel_dark,sw_absorbed_W_m2_Steel_dark_cool,sw_pv_W_m2_Steel_dark_cool
0,499.885417,321.244792,298.333333,51.914551,1.449219,821.130208,25.183333,2007-01-01 11:00:00,127.701333,739.017188,...,55.836854,21.844646,246.339063,96.373438,55.836854,21.844646,656.904167,256.995833,55.836854,21.844646
1,95.205729,384.854167,299.182292,46.557068,1.595703,480.059896,26.032292,2007-01-01 12:00:00,127.701333,432.053906,...,32.644073,26.170083,144.017969,115.456250,32.644073,26.170083,384.047917,307.883333,32.644073,26.170083
2,47.654948,366.979167,299.088542,49.282532,3.621094,414.634115,25.938542,2007-01-01 13:00:00,127.701333,373.170703,...,28.195120,24.954583,124.390234,110.093750,28.195120,24.954583,331.707292,293.583333,28.195120,24.954583
3,41.902344,404.229167,298.937500,48.566711,3.935547,446.131510,25.787500,2007-01-01 14:00:00,127.701333,401.518359,...,30.336943,27.487583,133.839453,121.268750,30.336943,27.487583,356.905208,323.383333,30.336943,27.487583
4,341.680990,400.179688,299.536458,45.318420,5.263672,741.860677,26.386458,2007-01-01 15:00:00,127.701333,667.674609,...,50.446526,27.212219,222.558203,120.053906,50.446526,27.212219,593.488542,320.143750,50.446526,27.212219


# External Convection Coefficient Calculation

Local windspeed calcuation - Converts BARRA2 wind speed from the meteorological measurement height to the local wind speed at the roof height. Utilises EnergyPlus Eq. 3.84

External convection coefficient calculation. Utilises EnergyPlus Eq. 3.104 McAdams Eq. Accounting to natural & forced convection


In [409]:
#LOCAL WINDSPEED CALCULATION 
# Site terrain values:  Current assumption: towns/cities
alpha = 0.33; delta = 460
# Meteorological station values:
z_met_m = 10 # assump = EnergyPlus default wind measurement height is 10 m
alpha_met = 0.14 # assump = open country 
delta_met = 270 # assump = open country 
# Calculate local wind speed at roof height and append to shortwave table
roof_sw_table["V_z_ms"] = roof_sw_table["wind_ms"]*(delta_met/z_met_m)**alpha_met*(z_roof_m/delta)**alpha
# External convection coefficient calculation and append to shortwave table
roof_sw_table["h_ext_Wm2K"] = 5.7+3.8*roof_sw_table["V_z_ms"]
display(roof_sw_table[["time_Melbourne","wind_ms","V_z_ms","h_ext_Wm2K"]].head())

,time_Melbourne,wind_ms,V_z_ms,h_ext_Wm2K
0,2007-01-01 11:00:00,1.449219,0.627646,8.085055
1,2007-01-01 12:00:00,1.595703,0.691087,8.326132
2,2007-01-01 13:00:00,3.621094,1.568269,11.659423
3,2007-01-01 14:00:00,3.935547,1.704457,12.176935
4,2007-01-01 15:00:00,5.263672,2.279658,14.362700


## Calculating intial midpoint temp in internal roof temps

Applied a steady state heat balance between the midpoints of each layer + boundaries
external -> steel = convection + shortwave + longwave (Black Body Radiation - using stephan boltzman eq.)
plaster -> steel = convection 
inner layers = conduction 

TSky = based humidity, time after midnight, 

In [410]:
# defining resistivity values for each roof layer
R2 = roof_layers_table["Cavity"].R_value_m2K_W; R3 = roof_layers_table["Insulation_old"].R_value_m2K_W; R4 = roof_layers_table["Plaster"].R_value_m2K_W
# storing first timestep of roof_sw_table into variables for use in matrix calculations
ho = roof_sw_table["h_ext_Wm2K"].iloc[0]
# ---------------------------------- defining sky temp --------------------------------
T_air_C = roof_sw_table["temp_C"].iloc[0]
T_air_K = T_air_C + 273.15
A1 = 17.625; B1 = 243.04; RH = roof_sw_table["rel_humidity_percent"].iloc[0]
T_dp_C = (B1*(math.log(RH/100) + (A1*T_air_C)/(B1+T_air_C))) / (A1-(math.log(RH/100) + (A1*T_air_C)/(B1+T_air_C))); t = roof_sw_table["time_Melbourne"].iloc[0].hour # time after midnight
T_sky_K = T_air_K*(0.711+0.0056*T_dp_C+0.000073*T_dp_C**2+0.013*math.cos(math.radians(15*t)))**0.25 # original sources uses radians, instead were doing degrees
# ------------------------------------------------------------------------------------
const = 5.6703e-8 # stephan boltzmann {W/m^2K^4}
# adaptive inside temp
if T_air_C > 18:
    T_indoor_K = T_indoor_C + 273.15
else: 
    T_indoor_K = 18 + 273.15
# adpative indoor convection - based on comparison of inside and outsdie temps + defining guess 
deltaT = T_air_K - T_indoor_K
if deltaT > 0:
    hi = 1.81*abs(deltaT)**(1/3)/(1.382+1)
    guess = [float(T_air_K+10), float(T_air_K-20), float(T_air_K-30), float(T_indoor_K-10), 0] # may need to reveiw the def of guess values   
else:
    hi = 9.482*abs(deltaT)**(1/3)/(7.238-1)
    guess = [float(T_air_K), float(T_air_K+20), float(T_air_K+30), float(T_indoor_K+10), 0] # may need to reveiw the def of guess values
print("Initial guess:", guess)
print("hi =", hi, "W/m^2K"); print("ho =", ho, "W/m^2K")
print("T_air_C =", T_air_C); print("T_dp_C =", T_dp_C); print("T_sky_C =", T_sky_K-273.15); print("T_indoor_C =", T_indoor_K-273.15, "C")
print("RH =", RH)
print("t =", t)
# empty list to store steady-state initial temperatures for each roof type
initial_temp_results = []
# eqautions dervied from steady state analysis of the roof layers and heat fluxes.
for roof_type in roof_types:
    roof = roof_layers_table[roof_type]
    R1 = roof.R_value_m2K_W
    emissivity = roof.Emissivity; emissivity_cool = roof.Emissivity_cool
    q_sw = roof_sw_table[f"sw_absorbed_W_m2_{roof_type}"].iloc[0]; q_sw_cool = roof_sw_table[f"sw_absorbed_W_m2_{roof_type}_cool"].iloc[0]
    def residuals(x,q_sw,emissivity): # hi is same for cool and normal roof - as intial guess is only based on inside to outsdie temp difference
        T1, T2, T3, T4, q = x # temps in kelvin
        F1 = hi*(T4 - T_indoor_K) - q # plaster -> indoor
        F2 = ho*(T_air_K - T1) + q_sw + emissivity*const*(T_sky_K**4 - T1**4) - q # outside -> first layer
        F3 = (2*(T1 - T2))/(R1 + R2) - q # first layer -> cavity
        F4 = (2*(T2 - T3))/(R2 + R3) - q # cavity -> ins
        F5 = (2*(T3 - T4))/(R3 + R4) - q # ins -> plaster
        return np.array([F1, F2, F3, F4, F5])
    sol = opt.root(residuals, guess, args=(q_sw,emissivity)) # iterative root solver
    sol_cool = opt.root(residuals, guess, args=(q_sw_cool,emissivity_cool))
    T1_K, T2_K, T3_K, T4_K, q_W_m2 = sol.x
    T1_K_cool, T2_K_cool, T3_K_cool, T4_K_cool, q_W_m2_cool = sol_cool.x
    # now to add accurate h values for the cavity - this is particularly important if outsdie temp < inside temp   
    dt = T1_K-T2_K; dt_cool = T1_K_cool - T2_K_cool
    if dt >= 0: # downwards heat flow top-> cavity mid 
        h_cav_a = (1.81*abs(dt)**(1/3))/(1.382+abs(math.cos(math.radians(roof_tilt)))) # energy plus eq. 3.156
    else:
        h_cav_a = (9.482*abs(dt)**(1/3))/(7.238-abs(math.cos(math.radians(roof_tilt)))) # energy plus eq. 3.155  
    if dt_cool >= 0:  # downwards heat flow top-> cavity mid 
        h_cav_a_cool = (1.81*abs(dt_cool)**(1/3))/(1.382+abs(math.cos(math.radians(roof_tilt))))
    else: 
        h_cav_a_cool = (9.482*abs(dt_cool)**(1/3))/(7.238-abs(math.cos(math.radians(roof_tilt)))) 
    dt = T2_K-T3_K; dt_cool = T2_K_cool-T3_K_cool
    if dt >= 0:  # downwards heat flow cavity-mid -> ins
        h_cav_b = (1.81*abs(dt)**(1/3))/(1.382+1)  # energy plus eq. 3.156
    else: 
        h_cav_b = (9.482*abs(dt)**(1/3))/(7.238-1)  # energy plus eq. 3.155
    if dt_cool >= 0:  # downwards heat flow cavity-mid -> ins
        h_cav_b_cool = (1.81*abs(dt_cool)**(1/3))/(1.382+1) 
    else: 
        h_cav_b_cool = (9.482*abs(dt_cool)**(1/3))/(7.238-1) 
    dt = T4_K - T_indoor_K; dt_cool = T4_K_cool - T_indoor_K
    if dt >= 0:  # downwards heat flow cavity-mid -> ins
        hi = (1.81*abs(dt)**(1/3))/(1.382+1)  # energy plus eq. 3.156
    else: 
        hi = (9.482*abs(dt)**(1/3))/(7.238-1)  # energy plus eq. 3.155
    if dt_cool >= 0:  # downwards heat flow cavity-mid -> ins
        hi_cool = (1.81*abs(dt_cool)**(1/3))/(1.382+1)  # energy plus eq. 3.156
    else: 
        hi_cool = (9.482*abs(dt_cool)**(1/3))/(7.238-1)  # energy plus eq. 3.155
    print(f"{roof_type} h_cav_a = {h_cav_a} & h_cav_b = {h_cav_b} & hi = {hi}\n Coolroof {roof_type} h_cav_a = {h_cav_a_cool} & h_cav_b = {h_cav_b_cool} & hi_cool = {hi_cool}")
    def residuals_2(x, q_sw,emissivity,hi,ha, hb):
            T1, T2, T3, T4, q = x # temps in kelvin
            F1 = hi*(T4 - T_indoor_K) - q # plaster -> indoor
            F2 = ho*(T_air_K - T1) + q_sw + emissivity*const*(T_sky_K**4 - T1**4) - q # outside -> first layer
            F3 = (T1 - T2) / ((1/h_cav_a) + (R1/2))- q # first layer -> cavity
            F4 = (T2 - T3) / ((1/h_cav_b)+ (R3/2)) - q # cavity -> ins
            F5 = (2*(T3 - T4))/(R3 + R4) - q # ins -> plaster
            return np.array([F1, F2, F3, F4, F5])
    sol = opt.root(residuals_2, guess, args=(q_sw,emissivity,hi, h_cav_a, h_cav_b)) # iterative root solver
    sol_cool = opt.root(residuals_2, guess, args=(q_sw_cool,emissivity_cool,hi_cool, h_cav_a_cool, h_cav_b_cool))
    T1_K, T2_K, T3_K, T4_K, q_W_m2 = sol.x
    T1_K_cool, T2_K_cool, T3_K_cool, T4_K_cool, q_W_m2_cool = sol_cool.x
    initial_temp_results.append({
        "Roof_type":roof_type,
        "T1_roof_C":T1_K-273.15, "T2_cavity_C":T2_K-273.15, "T3_insulation_C":T3_K-273.15, "T4_plaster_C":T4_K-273.15, "q_W_m2":q_W_m2,
        "T1_roof_K":T1_K, "T2_cavity_K":T2_K, "T3_insulation_K":T3_K, "T4_plaster_K":T4_K,
        "T1_roof_C_cool":T1_K_cool-273.15, "T2_cavity_C_cool":T2_K_cool-273.15, "T3_insulation_C_cool":T3_K_cool-273.15, "T4_plaster_C_cool":T4_K_cool-273.15,"q_W_m2_cool":q_W_m2_cool,
        "T1_roof_K_cool":T1_K_cool, "T2_cavity_K_cool":T2_K_cool, "T3_insulation_K_cool":T3_K_cool, "T4_plaster_K_cool":T4_K_cool})
# convert results into table
initial_temp_table = pd.DataFrame(initial_temp_results)
# show initial temperatures for each roof type
display(initial_temp_table)

Initial guess: [308.33333332999996, 278.33333332999996, 268.33333332999996, 283.15, 0]
hi = 1.3150427274368892 W/m^2K
ho = 8.085055072079417 W/m^2K
T_air_C = 25.18333333
T_dp_C = 14.607013670023042
T_sky_C = 8.626780681027526
T_indoor_C = 20.0 C
RH = 51.91455078
t = 11
Slate h_cav_a = 0.9893971029481399 & h_cav_b = 2.0236655221745425 & hi = 1.6511356940536135
 Coolroof Slate h_cav_a = 0.5324085440131119 & h_cav_b = 1.0889629765642845 & hi_cool = 0.8884994187063492
Terracotta h_cav_a = 0.9423895363542594 & h_cav_b = 1.8971815970986505 & hi = 1.4348460375434264
 Coolroof Terracotta h_cav_a = 0.548350495187217 & h_cav_b = 1.1039176774540014 & hi_cool = 0.8348973591623245
Concrete h_cav_a = 0.9043143245133028 & h_cav_b = 1.8382127449039396 & hi = 1.4568605839271547
 Coolroof Concrete h_cav_a = 0.407566958051612 & h_cav_b = 0.8284672224953061 & hi_cool = 0.6565949696927489
Steel_light h_cav_a = 0.6954004302644138 & h_cav_b = 1.4270508802827917 & hi = 1.125271833093743
 Coolroof Steel_light 

,Roof_type,T1_roof_C,T2_cavity_C,T3_insulation_C,T4_plaster_C,q_W_m2,T1_roof_K,T2_cavity_K,T3_insulation_K,T4_plaster_K,T1_roof_C_cool,T2_cavity_C_cool,T3_insulation_C_cool,T4_plaster_C_cool,q_W_m2_cool,T1_roof_K_cool,T2_cavity_K_cool,T3_insulation_K_cool,T4_plaster_K_cool
0,Slate,68.790907,58.176077,39.885637,26.351216,10.486719,341.940907,331.326077,313.035637,299.501216,27.620312,26.129074,23.559520,21.658121,1.473239,300.770312,299.279074,296.709520,294.808121
1,Terracotta,58.612537,50.066764,35.873834,25.566157,7.986578,331.762537,323.216764,309.023834,298.716157,27.623596,26.094707,23.555508,21.711403,1.428846,300.773596,299.244707,296.705508,294.861403
2,Concrete,55.995016,47.808494,34.579084,25.061724,7.374226,329.145016,320.958494,307.729084,298.211724,23.305105,22.663394,21.626389,20.880358,0.578038,296.455105,295.813394,294.776389,294.030358
3,Steel_light,36.808388,32.467409,26.578672,22.682650,3.018710,309.958388,305.617409,299.728672,295.832650,23.308507,22.542129,21.502503,20.814680,0.532938,296.458507,295.692129,294.652503,293.964680
4,Steel_dark,63.756924,53.908235,37.534722,25.584282,9.259421,336.906924,327.058235,310.684722,298.734282,23.303812,22.670751,21.618283,20.850124,0.595183,296.453812,295.820751,294.768283,294.000124


## Transient Heat Ingress Model
Outputting hourly q value transmitted from plaster to indoor

To add: floating indoor temp range in which neitehr cooling or heating is applied


In [411]:
numHours =24#4*365
dt = 60 # seconds
numTimesteps = int(numHours*3600/dt)

# common layer order after the outside roof material
common_layer_order = ["Cavity", "Insulation_old", "Plaster"] # this could be expanded if we have old and new houses!
numLayers = 4
innerLayerId = numLayers-1
T_indoor_K = T_indoor_C+273.15

# dictionaries to store the properties and initial temperatures for each roof type
Cp_by_roof = {}
Thicknesses_by_roof = {}
R_values_by_roof = {}
density_by_roof = {}
MidTemps_by_roof = {}

for roof_type in roof_types:
    layer_order = [roof_type] + common_layer_order
    layers = [roof_layers_table[layer] for layer in layer_order]
    Cp_by_roof[roof_type] = np.array([layer.Heat_capacity_J_KgK for layer in layers], dtype=float)
    Thicknesses_by_roof[roof_type] = np.array([layer.Thickness_m for layer in layers], dtype=float)
    R_values_by_roof[roof_type] = np.array([layer.R_value_m2K_W for layer in layers], dtype=float)
    density_by_roof[roof_type] = np.array([layer.Density_Kg_m3 for layer in layers], dtype=float)
    initial = initial_temp_table[initial_temp_table["Roof_type"] == roof_type].iloc[0]
    MidTemps_by_roof[roof_type] = np.array([initial["T1_roof_K"], initial["T2_cavity_K"], initial["T3_insulation_K"], initial["T4_plaster_K"]], dtype=float)




## Determining Maximum Timestep for Given Time Period

Forward Euler must resolve the fastest layer energy balance. The cavity is limiting because its capacity is small relative to its two conductances

The timestep limit is calculated for every layer & the smallest value is selected as the maximum allowable timestep for the entire simulation

In [412]:
# Using maximum convection coefficients for a conservative timestep check. This does not change the R_values used by the transient simulation.
# maximum outside convection coefficient over the simulation period
ho_max = float(roof_sw_table["h_ext_Wm2K"].iloc[:numHours+1].max())
# assumed maximum temperature difference for the adaptive convection correlations
deltaT_max = 5
# maximum indoor convection coefficient - downward heat flow gives the larger value
hi_max = 9.482*(abs(deltaT_max)**(1/3))/(7.238-1) # EnergyPlus eq (3.155)
# maximum cavity convection coefficient at roof -> cavity interface
# upward heat flow gives the largest h value
h_cav_a_max = (9.482*abs(deltaT_max)**(1/3))/(7.238-abs(math.cos(math.radians(roof_tilt))))
# maximum cavity convection coefficient at cavity -> insulation interface
h_cav_b_max = (9.482*abs(deltaT_max)**(1/3))/(7.238-1) # insulation is horizontal so tilt = 0 deg cos(0)=1

timestep_results = []
for roof_type in roof_types:
    Cp = Cp_by_roof[roof_type]
    Thicknesses = Thicknesses_by_roof[roof_type]
    R_values = R_values_by_roof[roof_type]
    density = density_by_roof[roof_type]
    layer_order = [roof_type, "Cavity", "Insulation_new", "Plaster"]
    # conductance between roof layer and cavity
    G_roof_cavity = 1/(R_values[0]/2 + 1/h_cav_a_max)
    # conductance between cavity and insulation
    G_cavity_ins = 1/(R_values[2]/2 + 1/h_cav_b_max)
    # conductance between insulation and plaster
    G_ins_plaster = 2/(R_values[2] + R_values[3])
    # conductance between plaster and indoor air
    G_plaster_indoor = 1/(R_values[3]/2 + 1/hi_max)
    # total conductance acting on each layer
    layer_conductance_sum = np.array([
        ho_max + G_roof_cavity,
        G_roof_cavity + G_cavity_ins,
        G_cavity_ins + G_ins_plaster,
        G_ins_plaster + G_plaster_indoor])
    # maximum allowable Forward Euler timestep for each layer
    dt_limits = (density*Cp*Thicknesses)/layer_conductance_sum
    # identify the limiting layer for this roof type
    limiting_layer_id = int(np.argmin(dt_limits))
    dt_max_positive = dt_limits[limiting_layer_id]

    timestep_results.append({
        "Roof_type": roof_type,
        "hi_max_W_m2K": hi_max,
        "h_cav_a_max_W_m2K": h_cav_a_max,
        "h_cav_b_max_W_m2K": h_cav_b_max,
        "Roof_dt_limit_s": dt_limits[0],
        "Cavity_dt_limit_s": dt_limits[1],
        "Insulation_dt_limit_s": dt_limits[2],
        "Plaster_dt_limit_s": dt_limits[3],
        "Limiting_layer": layer_order[limiting_layer_id],
        "Maximum_dt_s": dt_max_positive})
# convert timestep results into a table
timestep_limit_table = pd.DataFrame(timestep_results)
display(timestep_limit_table)

# find the most restrictive timestep across all roof types
overall_limiting_case = timestep_limit_table.loc[timestep_limit_table["Maximum_dt_s"].idxmin()]
print(f"ho_max = {ho_max:.2f} W/m^2K")
print(f"hi_max = {hi_max:.2f} W/m^2K")
print(f"h_cav_a_max = {h_cav_a_max:.2f} W/m^2K")
print(f"h_cav_b_max = {h_cav_b_max:.2f} W/m^2K")
print("Overall limiting roof type =", overall_limiting_case["Roof_type"])
print("Overall limiting layer =", overall_limiting_case["Limiting_layer"])
print(f"Maximum allowable timestep: dt < {overall_limiting_case['Maximum_dt_s']:.2f} s")

,Roof_type,hi_max_W_m2K,h_cav_a_max_W_m2K,h_cav_b_max_W_m2K,Roof_dt_limit_s,Cavity_dt_limit_s,Insulation_dt_limit_s,Plaster_dt_limit_s,Limiting_layer,Maximum_dt_s
0,Slate,2.599229,2.574341,2.599229,701.212628,228.048924,945.08604,3318.831518,Cavity,228.048924
1,Terracotta,2.599229,2.574341,2.599229,1307.729707,231.523095,945.08604,3318.831518,Cavity,231.523095
2,Concrete,2.599229,2.574341,2.599229,1568.542176,229.389608,945.08604,3318.831518,Cavity,229.389608
3,Steel_light,2.599229,2.574341,2.599229,90.339653,227.342110,945.08604,3318.831518,Steel_light,90.339653
4,Steel_dark,2.599229,2.574341,2.599229,90.339653,227.342110,945.08604,3318.831518,Steel_dark,90.339653


ho_max = 14.83 W/m^2K
hi_max = 2.60 W/m^2K
h_cav_a_max = 2.57 W/m^2K
h_cav_b_max = 2.60 W/m^2K
Overall limiting roof type = Steel_light
Overall limiting layer = Steel_light
Maximum allowable timestep: dt < 90.34 s


## Transient Model

In [413]:
# Select plotting mode
DoPlots = numHours < 18
# Checks
dt_max_positive = timestep_limit_table["Maximum_dt_s"].min()
if dt >= dt_max_positive:
    raise ValueError(f"dt={dt}s is too large. Use dt<{dt_max_positive:.2f}s")

# override the default thicnkess value with adaptive value dependant on plan area and pitch value, assuming a rectangualr floor plan with 2:1 ratio
cavity_thickness_avg = (1/4)*np.sqrt(roof_plan_area/2)*np.tan(np.radians(roof_tilt))

# Storage + interpolated weather
roof_results = {}
source_times = np.arange(len(roof_sw_table))*3600-1800
timesteps_per_hour = int(np.ceil(3600/dt))
total_timesteps = np.arange(numHours*timesteps_per_hour)*dt
temp_series = np.interp(total_timesteps,source_times,roof_sw_table["temp_C"].to_numpy()+273.15)
ho_series = np.interp(total_timesteps,source_times,roof_sw_table["h_ext_Wm2K"].to_numpy())
RH_series = np.interp(total_timesteps,source_times,roof_sw_table["rel_humidity_percent"].to_numpy())
wind_series = np.interp(total_timesteps,source_times,roof_sw_table["V_z_ms"].to_numpy())

# PV parameters
emissivity_pv = 0.90; U0_pv = 25; U1_pv = 6.84; pv_coverage = 0.3

for roof_type in roof_types:
    #print("\nRunning:",roof_type)
    # Roof properties
    roof = roof_layers_table[roof_type]
    Cp = Cp_by_roof[roof_type]; Thicknesses = Thicknesses_by_roof[roof_type]; R_values = R_values_by_roof[roof_type]; density = density_by_roof[roof_type]
    emissivity = roof.Emissivity; absorptivity = roof.Absorptivity;  absorptivity_cool = roof.Absorptivity_cool; emissivity_cool = roof.Emissivity_cool; layer_order = [roof_type,"Cavity","Insulation_old","Plaster"]
    total_solar_series = np.interp(total_timesteps,source_times,roof_sw_table[f"sw_absorbed_W_m2_{roof_type}"].to_numpy())
    diffuse_solar_series = np.interp(total_timesteps,source_times,roof_sw_table[f"sw_pv_W_m2_{roof_type}"].to_numpy()) 
    cool_total_solar_series = np.interp(total_timesteps,source_times,roof_sw_table[f"sw_absorbed_W_m2_{roof_type}_cool"].to_numpy())
    cool_diffuse_solar_series = np.interp(total_timesteps,source_times,roof_sw_table[f"sw_pv_W_m2_{roof_type}"].to_numpy()) 
    
    # -------------------------------------------------------------------------------------------
    Thicknesses_by_roof[roof_type][1] = cavity_thickness_avg # right now i have applied this to every house type, but the final code will need to consdier the specific house type and plan area.
    # -------------------------------------------------------------------------------------------
    # Initial temperatures
    MidTemps = MidTemps_by_roof[roof_type].copy()
    cool_row = initial_temp_table.loc[initial_temp_table["Roof_type"]==roof_type].iloc[0]
    MidTemps_cool = cool_row[["T1_roof_K_cool","T2_cavity_K_cool","T3_insulation_K_cool","T4_plaster_K_cool"]].to_numpy(dtype=float)
    # Hourly storage
    hourly_pathway_C = np.zeros((numHours,11)); hourly_pathway_cool_C = np.zeros((numHours,11))
    hourly_heat_in_Wh = np.zeros(numHours); hourly_heat_in_cool_Wh = np.zeros(numHours); hourly_heat_saved_Wh = np.zeros(numHours)
    hourly_sw_Wh = np.zeros(numHours); hourly_sw_cool_Wh = np.zeros(numHours); hourly_sw_difference_Wh = np.zeros(numHours)
    hourly_conv_Wh = np.zeros(numHours); hourly_conv_cool_Wh = np.zeros(numHours); hourly_conv_difference_Wh = np.zeros(numHours)
    hourly_lw_Wh = np.zeros(numHours); hourly_lw_cool_Wh = np.zeros(numHours); hourly_lw_difference_Wh = np.zeros(numHours)
    hourly_q_values_W = np.zeros((numHours,numLayers+1)); hourly_q_values_cool_W = np.zeros((numHours,numLayers+1))
    # Geometry
    x_pathway = np.array([-0.1,0,Thicknesses[0]/2,Thicknesses[0],Thicknesses[0]+Thicknesses[1]/2,Thicknesses[0]+Thicknesses[1],Thicknesses[0]+Thicknesses[1]+Thicknesses[2]/2,Thicknesses[0]+Thicknesses[1]+Thicknesses[2],Thicknesses[0]+Thicknesses[1]+Thicknesses[2]+Thicknesses[3]/2,np.sum(Thicknesses),np.sum(Thicknesses)+0.1])
    layer_areas = np.array([actual_roof_area,roof_plan_area,roof_plan_area,roof_plan_area])
    # Initial indoor convection coefficients
    if T_air_C <= 18:
        T_indoor_K = 18 + 273.15 
    else:                    
        T_indoor_K = 20 + 273.15
    T_ceiling = MidTemps[-1]; T_ceiling_cool = MidTemps_cool[-1]
    if T_ceiling >= T_indoor_K:
        hi = 1.81*abs(T_ceiling-T_indoor_K)**(1/3)/(1.382+1)
    else:
        hi = 9.482*abs(T_ceiling-T_indoor_K)**(1/3)/(7.238-1)
    if T_ceiling_cool >= T_indoor_K:
        hi_cool = 1.81*abs(T_ceiling-T_indoor_K)**(1/3)/(1.382+1)
    else:
        hi_cool = 9.482*abs(T_ceiling-T_indoor_K)**(1/3)/(7.238-1)
    for hour in range(numHours):
        start = roof_sw_table["time_Melbourne"].iloc[hour]; t0 = start.hour+start.minute/60; times = t0+np.arange(timesteps_per_hour)*dt/3600
        heat_in_Wh = 0; heat_in_cool_Wh = 0; sw_Wh = 0; sw_cool_Wh = 0; conv_Wh = 0; conv_cool_Wh = 0; lw_Wh = 0; lw_cool_Wh = 0
        for timestep in range(timesteps_per_hour):
            n = hour*timesteps_per_hour+timestep
            T_air_K = temp_series[n]; T_air_C = T_air_K-273.15; ho = ho_series[n]; humidity = RH_series[n]; wind_speed = wind_series[n]
            if T_air_C <= 18:
                T_indoor_K = 18 + 273.15 
            else:
                T_indoor_K = 20 + 273.15
            humidity_term = math.log(humidity/100)+(A1*T_air_C)/(B1+T_air_C)
            T_dp_C = B1*humidity_term/(A1-humidity_term); T_sky_K = T_air_K*(0.711+0.0056*T_dp_C+0.000073*T_dp_C**2+0.013*math.cos(math.radians(15*times[timestep])))**0.25
            # ---------------- NORMAL ROOF EXTERNAL BOUNDARY ----------------
            q_sw_bare_W = total_solar_series[n]*roof_plan_area
            q_conv_bare_W = ho*(T_air_K-MidTemps[0])*actual_roof_area
            q_lw_bare_W = emissivity*const*(T_sky_K**4-MidTemps[0]**4)*actual_roof_area
            # ---------------- COOL ROOF EXTERNAL BOUNDARY ----------------
            q_sw_bare_cool_W = cool_total_solar_series[n]*roof_plan_area
            q_conv_bare_cool_W = ho*(T_air_K-MidTemps_cool[0])*actual_roof_area
            q_lw_bare_cool_W = emissivity_cool*const*(T_sky_K**4-MidTemps_cool[0]**4)*actual_roof_area
            # ---------------- PV ----------------
            G_pv_W_m2 = total_solar_series[n]/max(roof.Absorptivity,1e-6)
            T_pv_K = T_air_K+G_pv_W_m2/(U0_pv+U1_pv*wind_speed)
            delta_T_pv = T_pv_K-MidTemps[0] # normal roof 
            if delta_T_pv >= 0:
                h_pv_gap = 1.81*abs(delta_T_pv)**(1/3)/(1.382+1)
            else:
                h_pv_gap = 9.482*abs(delta_T_pv)**(1/3)/(7.238-1)
            delta_T_pv_cool = T_pv_K-MidTemps_cool[0] # cool roof
            if delta_T_pv_cool >= 0:
                h_pv_gap_cool = 1.81*abs(delta_T_pv_cool)**(1/3)/(1.382+1)
            else:
                h_pv_gap_cool = 9.482*abs(delta_T_pv_cool)**(1/3)/(7.238-1)
            q_conv_pv_W = h_pv_gap*delta_T_pv*actual_roof_area
            q_conv_pv_cool_W = h_pv_gap_cool*delta_T_pv_cool*actual_roof_area
            eps_eff = 1/(1/emissivity_pv+1/emissivity-1); eps_eff_cool = 1/(1/emissivity_pv+1/emissivity_cool-1)
            q_lw_pv_W = eps_eff*const*(T_pv_K**4-MidTemps[0]**4)*actual_roof_area
            q_lw_pv_cool_W = eps_eff_cool*const*(T_pv_K**4-MidTemps_cool[0]**4)*actual_roof_area
            q_sw_pv_W = diffuse_solar_series[n]*roof_plan_area
            q_sw_pv_cool_W = cool_diffuse_solar_series[n]*roof_plan_area
            # ---------------- COMBINE PV + EXPOSED AREA ----------------
            q_sw_W = (1-pv_coverage)*q_sw_bare_W + pv_coverage*q_sw_pv_W
            q_conv_W = (1-pv_coverage)*q_conv_bare_W + pv_coverage*q_conv_pv_W
            q_lw_W = (1-pv_coverage)*q_lw_bare_W + pv_coverage*q_lw_pv_W
            q_sw_cool_W = (1-pv_coverage)*q_sw_bare_cool_W + pv_coverage*q_sw_pv_W
            q_conv_cool_W = (1-pv_coverage)*q_conv_bare_cool_W + pv_coverage*q_conv_pv_cool_W
            q_lw_cool_W = (1-pv_coverage)*q_lw_bare_cool_W + pv_coverage*q_lw_pv_cool_W
            # ---------------- NORMAL INTERNAL HEAT FLOW --------------------------------------------------------
            q = np.zeros(numLayers+1); q[0] = q_sw_W+q_conv_W+q_lw_W
            delta_T_a = MidTemps[0]-MidTemps[1]
            if delta_T_a >= 0: # downwards heat flow - reduced convection 
                h_cav_a = 1.81*abs(delta_T_a)**(1/3)/(1.382+abs(math.cos(math.radians(roof_tilt)))); q[1] = delta_T_a/(R_values[0]/2+1/h_cav_a)*actual_roof_area
            else: # enhanced
                h_cav_a = 9.482*abs(delta_T_a)**(1/3)/(7.238-abs(math.cos(math.radians(roof_tilt)))); q[1] = delta_T_a/(R_values[0]/2+1/h_cav_a)*actual_roof_area
            delta_T_b = MidTemps[1]-MidTemps[2]
            if delta_T_b >= 0:
                h_cav_b = 1.81*abs(delta_T_b)**(1/3)/(1.382+1); q[2] = delta_T_b/(R_values[2]/2+1/h_cav_b)*roof_plan_area
            else:
                h_cav_b = 9.482*abs(delta_T_b)**(1/3)/(7.238-1); q[2] = delta_T_b/(R_values[2]/2+1/h_cav_b)*roof_plan_area
            q[3] = 2*(MidTemps[2]-MidTemps[3])/(R_values[2]+R_values[3])*roof_plan_area
            q[-1] = (MidTemps[-1]-T_indoor_K)/(R_values[-1]/2+1/hi)*roof_plan_area
            # ---------------- UPDATE hi ----------------
            T_ceiling = MidTemps[-1]-(q[-1]/roof_plan_area)*R_values[-1]/2
            delta_T_c = T_ceiling - T_indoor_K
            if delta_T_c >= 0:
                hi = 1.81*abs(delta_T_c)**(1/3)/(1.382+1)
            else:
                hi = 9.482*abs(delta_T_c)**(1/3)/(7.238-1)
            # ---------------- COOL INTERNAL HEAT FLOW ----------------------------------------------------------
            q_cool = np.zeros(numLayers+1); q_cool[0] = q_sw_cool_W+q_conv_cool_W+q_lw_cool_W
            delta_T_a_cool = MidTemps_cool[0]-MidTemps_cool[1]
            if delta_T_a_cool >= 0:
                h_cav_a_cool = 1.81*abs(delta_T_a_cool)**(1/3)/(1.382+abs(math.cos(math.radians(roof_tilt)))); q_cool[1] = delta_T_a_cool/(R_values[0]/2+1/h_cav_a_cool)*actual_roof_area
            else:
                h_cav_a_cool = 9.482*abs(delta_T_a_cool)**(1/3)/(7.238-abs(math.cos(math.radians(roof_tilt)))); q_cool[1] = delta_T_a_cool/(R_values[0]/2+1/h_cav_a_cool)*actual_roof_area
            delta_T_b_cool = MidTemps_cool[1]-MidTemps_cool[2]
            if delta_T_b_cool >= 0:
                h_cav_b_cool = 1.81*abs(delta_T_b_cool)**(1/3)/(1.382+1); q_cool[2] = delta_T_b_cool/(R_values[2]/2+1/h_cav_b_cool)*roof_plan_area
            else:
                h_cav_b_cool = 9.482*abs(delta_T_b_cool)**(1/3)/(7.238-1); q_cool[2] = (delta_T_b_cool/(R_values[2]/2) + h_cav_b_cool*delta_T_b_cool)*roof_plan_area
            q_cool[3] = 2*(MidTemps_cool[2]-MidTemps_cool[3])/(R_values[2]+R_values[3])*roof_plan_area
            q_cool[-1] = ((MidTemps_cool[-1]-T_indoor_K)/(R_values[-1]/2 + 1/hi_cool))*roof_plan_area
            # ---------------- UPDATE cool hi ----------------
            T_ceiling_cool = MidTemps_cool[-1]-(q_cool[-1]/roof_plan_area)*R_values[-1]/2
            delta_T_c_cool = T_ceiling_cool - T_indoor_K
            if delta_T_c_cool >= 0:
                hi_cool = 1.81*abs(delta_T_c_cool)**(1/3)/(1.382+1)
            else:
                hi_cool = 9.482*abs(delta_T_c_cool)**(1/3)/(7.238-1)
            # ---------------- ENERGY ACCUMULATION ----------------
            heat_in_Wh += q[-1]*dt/3600; heat_in_cool_Wh += q_cool[-1]*dt/3600
            sw_Wh += q_sw_W*dt/3600; sw_cool_Wh += q_sw_cool_W*dt/3600
            conv_Wh += q_conv_W*dt/3600; conv_cool_Wh += q_conv_cool_W*dt/3600
            lw_Wh += q_lw_W*dt/3600; lw_cool_Wh += q_lw_cool_W*dt/3600
            # ---------------- TRANSIENT UPDATE ----------------
            MidTemps += dt*(q[:-1]-q[1:])/(density*Cp*Thicknesses*layer_areas)
            MidTemps_cool += dt*(q_cool[:-1]-q_cool[1:])/(density*Cp*Thicknesses*layer_areas)
        # End-of-hour heat flows
        outdoor_temp_hourly = temp_series[(hour+1)*timesteps_per_hour-1]
        q_end = q.copy(); q_end_cool = q_cool.copy()
        # Interface temperatures
        outside_roof_surface = MidTemps[0]+(q[0]/actual_roof_area)*R_values[0]/2
        roof_cavity_surface = MidTemps[0]-(q_end[1]/actual_roof_area)*R_values[0]/2
        cavity_ins_surface = MidTemps[2]+(q_end[2]/roof_plan_area)*R_values[2]/2
        ins_plaster_surface = MidTemps[2]-(q_end[3]/roof_plan_area)*R_values[2]/2
        inside_surface = MidTemps[3]-(q_end[-1]/roof_plan_area)*R_values[3]/2
        outside_roof_surface_cool = MidTemps_cool[0]+(q_cool[0]/actual_roof_area)*R_values[0]/2
        roof_cavity_surface_cool = MidTemps_cool[0]-(q_end_cool[1]/actual_roof_area)*R_values[0]/2
        cavity_ins_surface_cool = MidTemps_cool[2]+(q_end_cool[2]/roof_plan_area)*R_values[2]/2
        ins_plaster_surface_cool = MidTemps_cool[2]-(q_end_cool[3]/roof_plan_area)*R_values[2]/2
        inside_surface_cool = MidTemps_cool[3]-(q_end_cool[-1]/roof_plan_area)*R_values[3]/2
        # Hourly storage
        hourly_pathway_C[hour] = np.r_[outdoor_temp_hourly,outside_roof_surface,MidTemps[0],roof_cavity_surface,MidTemps[1],cavity_ins_surface,MidTemps[2],ins_plaster_surface,MidTemps[3],inside_surface,T_indoor_K]-273.15
        hourly_pathway_cool_C[hour] = np.r_[outdoor_temp_hourly,outside_roof_surface_cool,MidTemps_cool[0],roof_cavity_surface_cool,MidTemps_cool[1],cavity_ins_surface_cool,MidTemps_cool[2],ins_plaster_surface_cool,MidTemps_cool[3],inside_surface_cool,T_indoor_K]-273.15
        hourly_heat_in_Wh[hour] = heat_in_Wh; hourly_heat_in_cool_Wh[hour] = heat_in_cool_Wh; hourly_heat_saved_Wh[hour] = heat_in_Wh-heat_in_cool_Wh
        hourly_sw_Wh[hour] = sw_Wh; hourly_sw_cool_Wh[hour] = sw_cool_Wh; hourly_sw_difference_Wh[hour] = sw_Wh-sw_cool_Wh
        hourly_conv_Wh[hour] = conv_Wh; hourly_conv_cool_Wh[hour] = conv_cool_Wh; hourly_conv_difference_Wh[hour] = conv_Wh-conv_cool_Wh
        hourly_lw_Wh[hour] = lw_Wh; hourly_lw_cool_Wh[hour] = lw_cool_Wh; hourly_lw_difference_Wh[hour] = lw_Wh-lw_cool_Wh
        hourly_q_values_W[hour] = q_end; hourly_q_values_cool_W[hour] = q_end_cool
    # Store completed results
    roof_results[roof_type] = {"hourly_pathway_C":hourly_pathway_C,"hourly_pathway_cool_C":hourly_pathway_cool_C,"hourly_heat_in_Wh":hourly_heat_in_Wh,"hourly_heat_in_cool_Wh":hourly_heat_in_cool_Wh,"hourly_heat_saved_Wh":hourly_heat_saved_Wh,"hourly_sw_Wh":hourly_sw_Wh,"hourly_sw_cool_Wh":hourly_sw_cool_Wh,"hourly_sw_difference_Wh":hourly_sw_difference_Wh,"hourly_conv_Wh":hourly_conv_Wh,"hourly_conv_cool_Wh":hourly_conv_cool_Wh,"hourly_conv_difference_Wh":hourly_conv_difference_Wh,"hourly_lw_Wh":hourly_lw_Wh,"hourly_lw_cool_Wh":hourly_lw_cool_Wh,"hourly_lw_difference_Wh":hourly_lw_difference_Wh,"hourly_q_values_W":hourly_q_values_W,"hourly_q_values_cool_W":hourly_q_values_cool_W,"final_MidTemps_K":MidTemps.copy(),"final_MidTemps_cool_K":MidTemps_cool.copy()}
    # Display hourly normal vs cool results
    # -------------------------------- plotting --------------------------------
    if DoPlots:
        from matplotlib.lines import Line2D
        plot_hours = min(12,numHours); plot_times = pd.to_datetime(roof_sw_table["time_Melbourne"].iloc[:plot_hours])
        display(pd.DataFrame({"SW_normal_Wh":hourly_sw_Wh[:plot_hours],"SW_cool_Wh":hourly_sw_cool_Wh[:plot_hours],"SW_difference_Wh":hourly_sw_difference_Wh[:plot_hours],
            "Conv_normal_Wh":hourly_conv_Wh[:plot_hours],"Conv_cool_Wh":hourly_conv_cool_Wh[:plot_hours],"Conv_difference_Wh":hourly_conv_difference_Wh[:plot_hours],
            "LW_normal_Wh":hourly_lw_Wh[:plot_hours],"LW_cool_Wh":hourly_lw_cool_Wh[:plot_hours],"LW_difference_Wh":hourly_lw_difference_Wh[:plot_hours],
            "Heat_in_normal_Wh":hourly_heat_in_Wh[:plot_hours],"Heat_in_cool_Wh":hourly_heat_in_cool_Wh[:plot_hours],"Heat_saved_Wh":hourly_heat_saved_Wh[:plot_hours]},index=plot_times))
        layer_boundaries = np.r_[0,np.cumsum(Thicknesses)]
        fig,(ax_normal,ax_cool)=pl.subplots(2,1,figsize=(15,12),sharex=True,sharey=True)
    for h in range(plot_hours):
        line,=ax_normal.plot(x_pathway[1:10],hourly_pathway_C[h,1:10],linewidth=2,marker="o",markersize=4,label=plot_times.iloc[h].strftime("%I:%M %p").lstrip("0"))
        colour=line.get_color()
        ax_normal.plot(x_pathway[0:2],hourly_pathway_C[h,0:2],linestyle=":",linewidth=0.8,marker="o",markersize=3,color=colour)
        ax_normal.plot(x_pathway[9:11],hourly_pathway_C[h,9:11],linestyle=":",linewidth=0.8,marker="o",markersize=3,color=colour)

        ax_cool.plot(x_pathway[1:10],hourly_pathway_cool_C[h,1:10],linewidth=2,marker="o",markersize=4,color=colour)
        ax_cool.plot(x_pathway[0:2],hourly_pathway_cool_C[h,0:2],linestyle=":",linewidth=0.8,marker="o",markersize=3,color=colour)
        ax_cool.plot(x_pathway[9:11],hourly_pathway_cool_C[h,9:11],linestyle=":",linewidth=0.8,marker="o",markersize=3,color=colour)
    for ax in [ax_normal,ax_cool]:
        for boundary in layer_boundaries:
            ax.axvline(boundary,color="grey",linestyle="--",linewidth=.8)
        midpoint_positions=(layer_boundaries[:-1]+layer_boundaries[1:])/2
        for midpoint in midpoint_positions:
            ax.axvline(midpoint,color="grey",linestyle=":",linewidth=.5)
    label_positions=x_pathway.copy()
    x_labels=["Outside",f"{roof_type} surf",f"{roof_type} mid",f"{roof_type}/cavity","cavity mid","Cavity/ins","ins mid","ins/plaster","Plaster mid","Inside surface","Inside"]
    if roof_type in ["Steel_light","Steel_dark"]:
        x_labels=["Outside","",roof_type,"","cavity mid","Cavity/ins","ins mid","ins/plaster","Plaster mid","Inside surface","Inside"]
        label_positions[1]-=0.001; label_positions[3]+=0.001
    elif roof_type in ["Concrete","Terracotta"]:
        label_positions[1]-=0.001; label_positions[3]+=0.001
    else:
        label_positions[1]-=0.005; label_positions[3]+=0.005
    label_positions[7]-=0.002; label_positions[9]+=0.002
    for ax in [ax_normal,ax_cool]:
        ax.set_xticks(label_positions)
        ax.set_xticklabels(x_labels,rotation=90,ha="center",va="top",fontsize=9)
        for i,name in enumerate(layer_order):
            x_text=(layer_boundaries[i]+layer_boundaries[i+1])/2
            if name=="Plaster":
                x_text+=0.015
            ax.text(x_text,1.02,name,transform=ax.get_xaxis_transform(),ha="center",fontweight="bold")
        ax.set_xlim(x_pathway[0]-0.02,x_pathway[-1]+0.02)
        ax.grid(axis="y",alpha=.3)
    albedo=1-absorptivity
    albedo_cool=1-absorptivity_cool
    ax_normal.set_title(f"Normal roof  ($ε$={emissivity:.2f}, $α$={absorptivity:.2f}, albedo={albedo:.2f})",fontsize=13,pad=25)
    ax_cool.set_title(f"Cool roof  ($ε$={emissivity_cool:.2f}, $α$={absorptivity_cool:.2f}, albedo={albedo_cool:.2f})",fontsize=13,pad=25)
    ax_normal.text(.5,-.18,f"Average cavity thickness = {cavity_thickness_avg:.3f} m ",transform=ax_normal.transAxes,ha="center",va="top",fontsize=10)
    ax_normal.set_ylabel("Temperature (°C)")
    ax_cool.set_ylabel("Temperature (°C)")

    ax_cool.set_xlabel("Position through roof",labelpad=45)
    fig.suptitle(f"{roof_type} roof - {pv_coverage:.0%} PV coverage",fontsize=15,y=.98)
    handles,labels=ax_normal.get_legend_handles_labels()
    fig.legend(handles,labels,title="End time",bbox_to_anchor=(.90,.5),loc="center left")
    fig.subplots_adjust(bottom=.18,top=.91,right=.87,hspace=.38)
    pl.show()
# Final hourly heat-ingress comparison

hourly_energy_comparison = pd.DataFrame({"Time":roof_sw_table["time_Melbourne"].iloc[:numHours].to_numpy()})
for r in roof_types:
    normal = roof_results[r]["hourly_heat_in_Wh"]
    cool = roof_results[r]["hourly_heat_in_cool_Wh"]
    hourly_energy_comparison[f"{r}_normal_Wh"] = normal
    hourly_energy_comparison[f"{r}_cool_Wh"] = cool
    same_mode = np.sign(normal)==np.sign(cool)
    hourly_energy_comparison[f"{r}_saved_Wh"] = np.where(same_mode,np.abs(normal)-np.abs(cool),np.nan)
    hourly_energy_comparison[f"{r}_mode"] = np.select(
        [(normal>0)&(cool>0),(normal<0)&(cool<0),(normal>0)&(cool<0),(normal<0)&(cool>0)],
        ["Cooling","Heating","'Normal' cooling / 'Cool' heating","'Normal' heating / 'Cool' cooling"],
        default="No load")
display(hourly_energy_comparison)

,Time,Slate_normal_Wh,Slate_cool_Wh,Slate_saved_Wh,Slate_mode,Terracotta_normal_Wh,Terracotta_cool_Wh,Terracotta_saved_Wh,Terracotta_mode,Concrete_normal_Wh,...,Concrete_saved_Wh,Concrete_mode,Steel_light_normal_Wh,Steel_light_cool_Wh,Steel_light_saved_Wh,Steel_light_mode,Steel_dark_normal_Wh,Steel_dark_cool_Wh,Steel_dark_saved_Wh,Steel_dark_mode
0,2007-01-01 11:00:00,1048.746431,180.710758,868.035673,Cooling,865.812473,181.494634,684.317839,Cooling,774.775068,...,697.566506,Cooling,331.656653,74.897306,256.759347,Cooling,884.939124,86.769164,798.169960,Cooling
1,2007-01-01 12:00:00,1006.274500,226.245295,780.029205,Cooling,839.138729,207.128122,632.010608,Cooling,772.152266,...,667.479013,Cooling,330.214238,101.583967,228.630271,Cooling,845.082844,141.832927,703.249917,Cooling
2,2007-01-01 13:00:00,874.773526,257.766945,617.006580,Cooling,734.820816,230.211491,504.609325,Cooling,688.363926,...,547.801557,Cooling,305.894145,118.418565,187.475580,Cooling,748.143068,179.439655,568.703413,Cooling
3,2007-01-01 14:00:00,814.200279,278.862294,535.337985,Cooling,667.597401,245.396472,422.200929,Cooling,626.496995,...,457.878151,Cooling,299.740935,133.157858,166.583077,Cooling,720.625951,207.734340,512.891611,Cooling
4,2007-01-01 15:00:00,790.955602,288.490215,502.465388,Cooling,645.013150,256.266859,388.746291,Cooling,605.208518,...,416.863221,Cooling,295.128054,142.044996,153.083058,Cooling,691.911100,217.219035,474.692065,Cooling
5,2007-01-01 16:00:00,632.156559,247.272426,384.884133,Cooling,534.047629,227.686048,306.361580,Cooling,508.349836,...,332.572755,Cooling,246.020167,127.210491,118.809676,Cooling,549.095376,187.488747,361.606629,Cooling
6,2007-01-01 17:00:00,578.488858,223.064434,355.424424,Cooling,473.928574,204.921215,269.007359,Cooling,448.454153,...,286.848621,Cooling,234.156770,124.245597,109.911173,Cooling,518.362429,174.299915,344.062515,Cooling
7,2007-01-01 18:00:00,528.024994,198.616529,329.408466,Cooling,435.322184,185.978324,249.343860,Cooling,411.992987,...,263.438035,Cooling,213.089158,114.660784,98.428375,Cooling,464.906779,153.806158,311.100621,Cooling
8,2007-01-01 19:00:00,419.309183,158.460516,260.848667,Cooling,356.604516,153.366085,203.238431,Cooling,341.141042,...,216.562896,Cooling,168.935836,91.464792,77.471044,Cooling,359.169082,119.600855,239.568227,Cooling
9,2007-01-01 20:00:00,281.988388,101.692634,180.295754,Cooling,251.995226,105.564695,146.430531,Cooling,245.541647,...,157.643180,Cooling,110.780891,52.710952,58.069939,Cooling,231.804510,70.283107,161.521403,Cooling
